In [ ]:
!pip install -q gdown

!gdown --fuzzy "https://drive.google.com/file/d/15KWNlF0XM5QKYuAEop1oPA6wZdIXyLYp/view?usp=drive_link" -O panel_augmented_dataset.zip
!gdown --fuzzy "YOUR_OBELISK_GOOGLE_DRIVE_SHARE_LINK_HERE" -O obelisk_augmented_dataset.zip

In [ ]:
import zipfile, os
import yaml, glob, shutil
import subprocess
import torch 
from ultralytics import YOLO
from IPython.display import Image, display
import glob
import shutil

In [ ]:
def extract_zip(zip_path, extract_dir):
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        for member in z.infolist():
            fixed_name  = member.filename.replace('\\', '/')
            target_path = os.path.join(extract_dir, fixed_name)
            if fixed_name.endswith('/'):
                os.makedirs(target_path, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target_path), exist_ok=True)
                with z.open(member) as src, open(target_path, 'wb') as dst:
                    dst.write(src.read())
    print(f"Extracted: {zip_path} → {extract_dir}")

def find_dataset_dir(root):
    for dirpath, _, filenames in os.walk(root):
        if "data.yaml" in filenames:
            return dirpath
    return None

extract_zip("panel_augmented_dataset.zip",   "/kaggle/working/panel_raw")
extract_zip("obelisk_augmented_dataset.zip", "/kaggle/working/obelisk_raw")

PANEL_DS   = find_dataset_dir("/kaggle/working/panel_raw")
OBELISK_DS = find_dataset_dir("/kaggle/working/obelisk_raw")

if PANEL_DS is None:
    raise FileNotFoundError("data.yaml not found inside panel_raw")
if OBELISK_DS is None:
    raise FileNotFoundError("data.yaml not found inside obelisk_raw")

print(f"\nPanel dataset   : {PANEL_DS}")
print(f"Obelisk dataset : {OBELISK_DS}")

In [ ]:
with open(os.path.join(OBELISK_DS, "data.yaml")) as f:
    obelisk_yaml = yaml.safe_load(f)
with open(os.path.join(PANEL_DS, "data.yaml")) as f:
    panel_yaml = yaml.safe_load(f)


# Create combined folder structure to train both obelisk and panels
WORK_DIR = "/kaggle/working/combined_dataset"
for split in ("train", "val"):
    os.makedirs(f"{WORK_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{WORK_DIR}/labels/{split}", exist_ok=True)

def copy_images(src_dir, dst_dir, prefix=""):
    count = 0
    for img in glob.glob(f"{src_dir}/*.jpg") + glob.glob(f"{src_dir}/*.png"):
        shutil.copy2(img, os.path.join(dst_dir, prefix + os.path.basename(img)))
        count += 1
    return count

def copy_labels(src_dir, dst_dir, class_offset=0, prefix=""):
    count = 0
    for txt in glob.glob(f"{src_dir}/*.txt"):
        out_path = os.path.join(dst_dir, prefix + os.path.basename(txt))
        with open(txt) as f_in, open(out_path, "w") as f_out:
            for line in f_in:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                f_out.write(f"{int(parts[0]) + class_offset} {' '.join(parts[1:])}\n")
        count += 1
    return count

# class 0 used for obelisk 
for split in ("train", "val"):
    n = copy_images(f"{OBELISK_DS}/images/{split}", f"{WORK_DIR}/images/{split}", prefix="obelisk_")
    print(f"Obelisk {split} images: {n}")
    n = copy_labels(f"{OBELISK_DS}/labels/{split}", f"{WORK_DIR}/labels/{split}", class_offset=0, prefix="obelisk_")
    print(f"Obelisk {split} labels: {n}")

# classes from 1 to 9 used for panels
for split in ("train", "val"):
    n = copy_images(f"{PANEL_DS}/images/{split}", f"{WORK_DIR}/images/{split}", prefix="panel_")
    print(f"Panel   {split} images: {n}")
    n = copy_labels(f"{PANEL_DS}/labels/{split}", f"{WORK_DIR}/labels/{split}", class_offset=1, prefix="panel_")
    print(f"Panel   {split} labels: {n}")

# check for counts
for split in ("train", "val"):
    imgs = len(glob.glob(f"{WORK_DIR}/images/{split}/*"))
    lbls = len(glob.glob(f"{WORK_DIR}/labels/{split}/*.txt"))
    status = "✓" if imgs == lbls else "← MISMATCH"
    print(f"  {split}: images={imgs}  labels={lbls}  {status}")

# combine data yamel
combined_names = ["obelisk"] + panel_yaml.get("names", [])  

COMBINED_YAML = f"{WORK_DIR}/data.yaml"
combined_data = {
    "path"  : WORK_DIR,
    "train" : "images/train",
    "val"   : "images/val",
    "nc"    : len(combined_names),
    "names" : combined_names,
}
with open(COMBINED_YAML, "w") as f:
    yaml.dump(combined_data, f, default_flow_style=False, sort_keys=False)


In [ ]:
subprocess.run(["pip", "install", "ultralytics", "-q"], check=True)
print("Ultralytics installed.")

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU — enable it in Settings -> Accelerator -> GPU T4 x2")

In [ ]:
# cell for training yolo
RUNS_DIR = "/kaggle/working/runs"
MODEL    = "yolov26s.pt"
EPOCHS   = 100
IMG_SIZE = 640
BATCH    = 16      
RUN_NAME = "obelisk_panel_detector"

model   = YOLO(MODEL)
results = model.train(
    data     = COMBINED_YAML,
    epochs   = EPOCHS,
    imgsz    = IMG_SIZE,
    batch    = BATCH,
    patience = 20,
    device   = 0,
    project  = RUNS_DIR,
    name     = RUN_NAME,
    exist_ok = True,
    verbose  = True,
)

WEIGHTS_DIR = f"{RUNS_DIR}/{RUN_NAME}/weights"
print(f"\nTraining completed and best weight saved at best.pt ")

In [ ]:
best_model  = YOLO(f"{WEIGHTS_DIR}/best.pt")
val_results = best_model.val(data=COMBINED_YAML, imgsz=IMG_SIZE, device=0)

print("\nValidation Results")
print(f"mAP50 : {val_results.box.map50:.4f}")
print(f"mAP50-95  : {val_results.box.map:.4f}")
print(f"Precision : {val_results.box.mp:.4f}")
print(f"Recall: {val_results.box.mr:.4f}")

In [ ]:
# cell used for creating confusion matrix

cm_path = glob.glob(f"{RUNS_DIR}/{RUN_NAME}/confusion_matrix_normalized.png")
if cm_path:
    display(Image(cm_path[0]))
else:
    print(f"Confusion matrix not found")

In [ ]:
best_model.export(
    format   = "onnx",
    imgsz    = IMG_SIZE,
    opset    = 12,
    simplify = True,
)
onnx_path = f"{WEIGHTS_DIR}/best.onnx"
print(f"\nExport model ")

In [ ]:
shutil.copy(f"{WEIGHTS_DIR}/best.pt",   "/kaggle/working/best.pt")
shutil.copy(f"{WEIGHTS_DIR}/best.onnx", "/kaggle/working/best.onnx")

with open("/kaggle/working/classes.txt", "w") as f:
    for name in combined_names:
        f.write(name + "\n")